# FallDeWideo — Préparation fusion (ResNet-18 CSI + MobileNetV4 OpenPose)

## ⚠️ Important — comment récupérer vos résultats demain sans tout relancer

Ce notebook doit être exécuté via **"Save Version" → "Save & Run All (Commit)"** (en haut à droite de l'éditeur Kaggle), **pas** en cliquant juste sur "Run All" dans une session interactive.

**Pourquoi ça change tout** :
- Une session interactive (celle où vous voyez le code s'exécuter en direct) **se ferme si vous fermez l'onglet ou après un certain temps d'inactivité** → tout ce qui n'a pas été téléchargé est perdu.
- **"Save & Run All (Commit)"** lance le notebook **en arrière-plan sur les serveurs Kaggle**, indépendamment de votre navigateur. Vous pouvez fermer l'onglet, éteindre votre PC, revenir le lendemain : le run continue et se termine tout seul.
- Une fois terminé, les fichiers produits dans `/kaggle/working/` sont **attachés en permanence à cette "Version" du notebook**, consultables et téléchargeables depuis l'onglet **Output** de cette version précise — sans avoir besoin de rouvrir/relancer le notebook.

**Ce que vous devez faire concrètement** :
1. Collez ce notebook dans Kaggle, attachez le dataset FallDeWideo.
2. Cliquez **Save Version → Save & Run All (Commit)**.
3. Fermez l'onglet si vous voulez, revenez plus tard (demain).
4. Allez dans l'onglet **Versions** du notebook → cliquez sur la version terminée → **Output** → téléchargez les 2 fichiers zip listés en bas de ce notebook.

## Ce que produit ce notebook
- `resnet18_csi_fusion_ready.zip` — poids + preprocessing + split + définition du modèle + métriques
- `mobilenetv4_openpose_fusion_ready.zip` — idem pour OpenPose
- Les deux sont entraînés sur **le même split commun** (paires CSI↔OpenPose alignées par la colonne `pic`), condition indispensable pour la fusion.

## Gestion mémoire
Le pipeline traite **entièrement le CSI puis libère toute sa mémoire (RAM+GPU)** avant de commencer l'OpenPose. Les deux modèles/DataLoaders ne coexistent jamais en mémoire.


In [1]:
import os, gc, re, json, pickle, random, time, warnings, math, shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\u26a0\ufe0f Aucun GPU d\u00e9tect\u00e9 -- active un acc\u00e9l\u00e9rateur (Settings > Accelerator) avant de commiter.")

def mem_gb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e9

CLASSES = ["fall", "nfall1", "nfall2"]
LABEL2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS = {i: c for c, i in LABEL2IDX.items()}

T_STEPS, F_SUB = 50, 30
N_LINKS, N_ANT_KEEP = 3, 3
CSI_IN_CHANNELS = N_LINKS * N_ANT_KEEP          # 9
CSI_TOTAL_CHANNELS = CSI_IN_CHANNELS * 2        # 18 (amp+phase)

N_AFF, N_KPT = 19, 38
OP_IN_CHANNELS = N_AFF + N_KPT                  # 57
OP_H, OP_W = 36, 64

CSI_BATCH_SIZE = 64
CSI_EPOCHS = 250
CSI_LR = 3e-4
CSI_WEIGHT_DECAY = 1e-4
CSI_LABEL_SMOOTHING = 0.1
CSI_DROPOUT = 0.3

OP_BATCH_SIZE = 128
OP_EPOCHS = 250
OP_LR = 1e-3
OP_WEIGHT_DECAY = 5e-4
OP_LABEL_SMOOTHING = 0.1
OP_DROPOUT = 0.5
OP_MIXUP_ALPHA = 0.2

RUN_CSI = True
RUN_OPENPOSE = True

OUT_ROOT = Path("/kaggle/working/fusion_preparation")
CACHE_ROOT = OUT_ROOT / "cache"
DEF_ROOT = OUT_ROOT / "definitions"
CKPT_ROOT = OUT_ROOT / "checkpoints"
META_ROOT = OUT_ROOT / "metadata"
FINAL_ROOT = Path("/kaggle/working")   # zips finaux ici, a la racine de l'Output

for p in [OUT_ROOT, CACHE_ROOT, DEF_ROOT, CKPT_ROOT, META_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print(f"RAM: {mem_gb():.2f} Go")


Device: cuda
GPU: Tesla T4
RAM: 0.75 Go


## 1. Localisation du dataset

In [2]:
def find_dataset_root():
    candidates = [
        Path("/kaggle/input/falldewideoo"),
        Path("/kaggle/input/datasets/imanemokhtari/falldewideoo"),
    ]
    candidates += [p for p in Path("/kaggle/input").glob("*") if p.is_dir()]
    for root in candidates:
        if not root.exists():
            continue
        pk_files = list(root.rglob("full_0.pk"))
        has_csi = any("/csi/" in f.as_posix() for f in pk_files)
        has_op = any("/openpose/" in f.as_posix() for f in pk_files)
        if has_csi and has_op:
            return root
    raise FileNotFoundError("FallDeWideo introuvable -- verifie que le dataset est attache (Add Input).")

DATA_ROOT = find_dataset_root()
print("DATA_ROOT:", DATA_ROOT)

def file_number(path):
    m = re.search(r"full_(\d+)\.pk$", str(path))
    return int(m.group(1)) if m else 999999

all_pk = list(DATA_ROOT.rglob("full_*.pk"))

normal_csi = [p for p in all_pk if p.as_posix().endswith("/csi/csi/" + p.name)]
missing_csi = [p for p in all_pk if p.as_posix().endswith("/missing/missing/csi/" + p.name)]
csi_by_number = {file_number(p): p for p in normal_csi}
csi_by_number.update({file_number(p): p for p in missing_csi})
CSI_FILES = [csi_by_number[n] for n in sorted(csi_by_number)]

normal_op = [p for p in all_pk if p.as_posix().endswith("/openpose/openpose/" + p.name)]
missing_op = [p for p in all_pk if p.as_posix().endswith("/missing/missing/openpose/" + p.name)]
op_by_number = {file_number(p): p for p in normal_op}
op_by_number.update({file_number(p): p for p in missing_op})
OPENPOSE_FILES = [op_by_number[n] for n in sorted(op_by_number)]

print(f"CSI files: {len(CSI_FILES)} | OpenPose files: {len(OPENPOSE_FILES)}")
assert len(CSI_FILES) > 0 and len(OPENPOSE_FILES) > 0


DATA_ROOT: /kaggle/input/datasets/imanemokhtari/falldewideoo
CSI files: 10 | OpenPose files: 10


## 2. Alignement CSI ↔ OpenPose par identité exacte (`pic`)

On ne suppose jamais que les lignes sont alignées par position — on matche par la valeur de la colonne
`pic`, avec détection de doublons et de labels incohérents entre les deux modalités.

In [3]:
def normalize_pic(x):
    return re.sub(r"\\+", "/", str(x)).strip().lower()

def label_from_pic(pic):
    s = normalize_pic(pic)
    parts = s.split("/")
    for token in ("nfall1", "nfall2", "fall"):
        if token in parts:
            return token
    for token in ("nfall1", "nfall2", "fall"):
        if token in s:
            return token
    return None

pair_records = []
alignment_report = []
op_map = {file_number(p): p for p in OPENPOSE_FILES}

for csi_path in CSI_FILES:
    n = file_number(csi_path)
    if n not in op_map:
        continue
    op_path = op_map[n]
    print(f"Checking full_{n}.pk ...")

    cdf = pd.read_pickle(csi_path)
    odf = pd.read_pickle(op_path)
    if "pic" not in cdf.columns or "pic" not in odf.columns:
        raise KeyError(f"Colonne pic absente dans full_{n}.pk")

    c_pic = cdf["pic"].map(normalize_pic)
    o_pic = odf["pic"].map(normalize_pic)
    c_set, o_set = set(c_pic), set(o_pic)

    c_dup = int(c_pic.duplicated().sum())
    o_dup = int(o_pic.duplicated().sum())
    alignment_report.append({
        "file_number": n, "csi_rows": len(cdf), "openpose_rows": len(odf),
        "common_pic": len(c_set & o_set), "csi_only": len(c_set - o_set),
        "openpose_only": len(o_set - c_set),
        "csi_duplicate_pic": c_dup, "openpose_duplicate_pic": o_dup,
    })
    if c_dup or o_dup:
        raise ValueError(f"Doublons pic pour full_{n}: CSI={c_dup}, OpenPose={o_dup}")

    o_lookup = {pic: i for i, pic in enumerate(o_pic)}
    for c_idx, pic in enumerate(c_pic):
        if pic not in o_lookup:
            continue
        o_idx = o_lookup[pic]
        lab_c = label_from_pic(cdf.iloc[c_idx]["pic"])
        lab_o = label_from_pic(odf.iloc[o_idx]["pic"])
        if lab_c is None or lab_o is None:
            continue
        if lab_c != lab_o:
            raise ValueError(f"Label mismatch full_{n}, pic={pic}: CSI={lab_c}, OP={lab_o}")
        pair_records.append({
            "file_number": n, "csi_row": c_idx, "openpose_row": o_idx,
            "pic": pic, "label": lab_c, "label_id": LABEL2IDX[lab_c],
        })
    del cdf, odf, c_pic, o_pic
    gc.collect()

pairs = pd.DataFrame(pair_records)
alignment_df = pd.DataFrame(alignment_report)
print(alignment_df.to_string(index=False))
print("\nPaires alignees:", len(pairs))
print(pairs["label"].value_counts())

pairs.to_csv(META_ROOT / "paired_samples.csv", index=False)
alignment_df.to_csv(META_ROOT / "alignment_report.csv", index=False)
assert len(pairs) > 0 and pairs["label"].notna().all()
print("\u2705 Alignement CSI/OpenPose valide.")


Checking full_0.pk ...
Checking full_1.pk ...
Checking full_2.pk ...
Checking full_3.pk ...
Checking full_4.pk ...
Checking full_5.pk ...
Checking full_6.pk ...
Checking full_7.pk ...
Checking full_8.pk ...
Checking full_9.pk ...
 file_number  csi_rows  openpose_rows  common_pic  csi_only  openpose_only  csi_duplicate_pic  openpose_duplicate_pic
           0      4020           4020        4020         0              0                  0                       0
           1      4020           4020        4020         0              0                  0                       0
           2      4020           4020        4020         0              0                  0                       0
           3      4020           4020        4020         0              0                  0                       0
           4      4020           4020        4020         0              0                  0                       0
           5      4020           4020        4020         0   

## 3. Split commun (70/15/15 stratifié, seed 42) — utilisé identiquement par les deux modalités

In [4]:
split_ids = np.arange(len(pairs))
train_ids, temp_ids = train_test_split(split_ids, test_size=0.30, stratify=pairs["label_id"].values, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, stratify=pairs.iloc[temp_ids]["label_id"].values, random_state=SEED)

pairs["split"] = "unused"
pairs.loc[train_ids, "split"] = "train"
pairs.loc[val_ids, "split"] = "val"
pairs.loc[test_ids, "split"] = "test"

for split_name in ["train", "val", "test"]:
    sub = pairs[pairs["split"] == split_name]
    print(split_name, len(sub), sub["label"].value_counts().to_dict())
    assert sub["pic"].is_unique

pairs.to_csv(META_ROOT / "paired_samples_with_split.csv", index=False)
np.savez(META_ROOT / "common_split.npz",
         train_ids=np.asarray(train_ids, dtype=np.int64),
         val_ids=np.asarray(val_ids, dtype=np.int64),
         test_ids=np.asarray(test_ids, dtype=np.int64))
with open(META_ROOT / "label_mapping.json", "w") as f:
    json.dump({"label2idx": LABEL2IDX, "idx2class": IDX2CLASS}, f, indent=2)
print("\n\u2705 Split commun sauvegarde.")


train 28140 {'nfall1': 10500, 'nfall2': 10010, 'fall': 7630}
val 6030 {'nfall1': 2250, 'nfall2': 2145, 'fall': 1635}
test 6030 {'nfall1': 2250, 'nfall2': 2145, 'fall': 1635}

✅ Split commun sauvegarde.


## 4. Prétraitement + cache CSI (mmap, `float16`)

Identique au notebook CSI fourni : moyenne d'antenne, amplitude, phase `unwrap` + retrait de tendance
linéaire, concaténation des 3 liens.

In [5]:
CSI_CACHE = CACHE_ROOT / "csi"
CSI_CACHE.mkdir(parents=True, exist_ok=True)

def sanitize_phase(phase):
    phase = np.unwrap(phase, axis=1)
    Fdim = phase.shape[1]
    idx = np.arange(Fdim, dtype=np.float32)
    idx_c = idx - idx.mean()
    denom = (idx_c ** 2).sum()
    phase_mean = phase.mean(axis=1, keepdims=True)
    phase_c = phase - phase_mean
    slope = (phase_c * idx_c[None, :, None]).sum(axis=1, keepdims=True) / denom
    return phase - slope * idx_c[None, :, None] - phase_mean

def process_link(arr_complex):
    arr = arr_complex.mean(axis=-1)
    amp = np.abs(arr).astype(np.float32)
    phase = sanitize_phase(np.angle(arr).astype(np.float32))
    return amp, phase

def process_row(row):
    amps, phas = [], []
    for col in ("csi0", "csi1", "csi2"):
        a, p = process_link(row[col])
        amps.append(a); phas.append(p)
    return (np.concatenate(amps, axis=-1).astype(np.float16),
            np.concatenate(phas, axis=-1).astype(np.float16))

pairs["cache_pos"] = -1
csi_manifest = []

for csi_path in CSI_FILES:
    n = file_number(csi_path)
    target_idx = pairs.index[pairs["file_number"] == n].tolist()
    if not target_idx:
        continue
    target_rows = pairs.loc[target_idx, "csi_row"].astype(int).tolist()
    amp_path = CSI_CACHE / f"AMP_{n}.npy"
    pha_path = CSI_CACHE / f"PHA_{n}.npy"

    if not (amp_path.exists() and pha_path.exists()):
        print(f"Pretraitement CSI full_{n} ...")
        df = pd.read_pickle(csi_path)
        amp_list, pha_list = [], []
        for row_idx in target_rows:
            a, p = process_row(df.iloc[row_idx])
            amp_list.append(a); pha_list.append(p)
        np.save(amp_path, np.stack(amp_list).astype(np.float16))
        np.save(pha_path, np.stack(pha_list).astype(np.float16))
        del df, amp_list, pha_list
        gc.collect()
    else:
        print(f"Cache CSI deja present: full_{n}")

    for pos, global_idx in enumerate(target_idx):
        pairs.loc[global_idx, "cache_pos"] = pos
    csi_manifest.append({"file_number": n, "amp_path": str(amp_path), "pha_path": str(pha_path), "n_samples": len(target_rows)})

assert (pairs["cache_pos"] >= 0).all()
with open(CSI_CACHE / "manifest.pkl", "wb") as f:
    pickle.dump(csi_manifest, f)
pairs.to_csv(META_ROOT / "paired_samples_with_cache.csv", index=False)
print(f"Fichiers CSI caches: {len(csi_manifest)} | RAM: {mem_gb():.2f} Go")


Pretraitement CSI full_0 ...
Pretraitement CSI full_1 ...
Pretraitement CSI full_2 ...
Pretraitement CSI full_3 ...
Pretraitement CSI full_4 ...
Pretraitement CSI full_5 ...
Pretraitement CSI full_6 ...
Pretraitement CSI full_7 ...
Pretraitement CSI full_8 ...
Pretraitement CSI full_9 ...
Fichiers CSI caches: 10 | RAM: 4.52 Go


## 5. Normalisation CSI (statistiques train uniquement)

In [6]:
csi_manifest_map = {m["file_number"]: m for m in csi_manifest}
train_pair_ids = pairs.index[pairs["split"] == "train"].to_numpy()
rng = np.random.default_rng(SEED)
stat_ids = rng.choice(train_pair_ids, size=min(3000, len(train_pair_ids)), replace=False)

sum_x, sum_x2, n_values = 0.0, 0.0, 0
for global_idx in stat_ids:
    r = pairs.loc[global_idx]
    m = csi_manifest_map[int(r.file_number)]
    amp_mm = np.load(m["amp_path"], mmap_mode="r")
    a = np.asarray(amp_mm[int(r.cache_pos)], dtype=np.float32)
    sum_x += float(a.sum()); sum_x2 += float(np.square(a).sum()); n_values += a.size

AMP_MEAN = sum_x / n_values
AMP_STD = float(np.sqrt(max(sum_x2 / n_values - AMP_MEAN**2, 0.0)) + 1e-6)
print("AMP_MEAN =", AMP_MEAN, "| AMP_STD =", AMP_STD)

with open(META_ROOT / "csi_normalization.json", "w") as f:
    json.dump({"AMP_MEAN": AMP_MEAN, "AMP_STD": AMP_STD, "phase_divisor": float(np.pi),
               "computed_from": "common_train_split_only", "seed": SEED}, f, indent=2)


AMP_MEAN = 17.412203257330248 | AMP_STD = 12.322658442598982


## 6. Dataset + DataLoaders CSI (mmap **caché en mémoire par fichier** — pas de réouverture par échantillon)

In [7]:
class CSIFusionPrepDataset(Dataset):
    def __init__(self, pairs_df, split, csi_manifest):
        self.df = pairs_df[pairs_df["split"] == split].reset_index(drop=True)
        self.manifest = {m["file_number"]: m for m in csi_manifest}
        self.split = split
        self._mmap_cache = {}

    def _get_mmap(self, file_number, key):
        ck = (file_number, key)
        if ck not in self._mmap_cache:
            self._mmap_cache[ck] = np.load(self.manifest[file_number][key], mmap_mode="r")
        return self._mmap_cache[ck]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        fn, pos = int(r.file_number), int(r.cache_pos)
        amp = np.asarray(self._get_mmap(fn, "amp_path")[pos], dtype=np.float32)
        pha = np.asarray(self._get_mmap(fn, "pha_path")[pos], dtype=np.float32)
        amp = (amp - AMP_MEAN) / AMP_STD
        pha = pha / np.pi
        if self.split == "train":
            amp = amp + np.random.normal(0, 0.02, amp.shape).astype(np.float32)
            if np.random.rand() < 0.3:
                t0 = np.random.randint(0, max(1, amp.shape[0] - 5))
                amp[t0:t0+5] = 0; pha[t0:t0+5] = 0
            if np.random.rand() < 0.3:
                f0 = np.random.randint(0, max(1, amp.shape[1] - 4))
                amp[:, f0:f0+4] = 0; pha[:, f0:f0+4] = 0
        x = np.concatenate([amp, pha], axis=-1)
        return torch.from_numpy(x), torch.tensor(int(r.label_id), dtype=torch.long)

train_csi = CSIFusionPrepDataset(pairs, "train", csi_manifest)
val_csi = CSIFusionPrepDataset(pairs, "val", csi_manifest)
test_csi = CSIFusionPrepDataset(pairs, "test", csi_manifest)

def make_sampler(pairs_df):
    labels = pairs_df[pairs_df["split"] == "train"]["label_id"].to_numpy()
    counts = np.bincount(labels, minlength=len(CLASSES))
    weights = 1.0 / np.maximum(counts, 1)
    return WeightedRandomSampler(torch.as_tensor(weights[labels], dtype=torch.double),
                                  num_samples=len(labels), replacement=True)

csi_sampler = make_sampler(pairs)
CSI_WORKERS = 2
csi_train_loader = DataLoader(train_csi, batch_size=CSI_BATCH_SIZE, sampler=csi_sampler,
                               num_workers=CSI_WORKERS, pin_memory=True, drop_last=True)
csi_val_loader = DataLoader(val_csi, batch_size=CSI_BATCH_SIZE, shuffle=False,
                             num_workers=CSI_WORKERS, pin_memory=True)
csi_test_loader = DataLoader(test_csi, batch_size=CSI_BATCH_SIZE, shuffle=False,
                              num_workers=CSI_WORKERS, pin_memory=True)
print("CSI:", len(train_csi), len(val_csi), len(test_csi))


CSI: 28140 6030 6030


## 7. ResNet-18 CSI — architecture identique au notebook fourni

In [8]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_planes != planes:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return self.relu(out)

class ResNet18_CSI(nn.Module):
    def __init__(self, in_channels=18, num_classes=3, dropout=0.3):
        super().__init__()
        self.in_planes = 64
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)
    def _make_layer(self, planes, blocks, stride):
        layers = [BasicBlock(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_planes, planes, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)

resnet18 = ResNet18_CSI(in_channels=CSI_TOTAL_CHANNELS, num_classes=len(CLASSES), dropout=CSI_DROPOUT).to(DEVICE)
n_params_resnet = sum(p.numel() for p in resnet18.parameters())
print(f"ResNet-18 CSI parametres: {n_params_resnet:,}")

RESNET18_SRC = '''import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_planes != planes:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return self.relu(out)

class ResNet18_CSI(nn.Module):
    def __init__(self, in_channels=18, num_classes=3, dropout=0.3):
        super().__init__()
        self.in_planes = 64
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)
    def _make_layer(self, planes, blocks, stride):
        layers = [BasicBlock(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_planes, planes, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)
'''
with open(DEF_ROOT / "resnet18_csi.py", "w") as f:
    f.write(RESNET18_SRC)
print("Definition exportee:", DEF_ROOT / "resnet18_csi.py")


ResNet-18 CSI parametres: 11,179,011
Definition exportee: /kaggle/working/fusion_preparation/definitions/resnet18_csi.py


## 8. Entraînement ResNet-18 CSI (AdamW, cosine annealing, label smoothing 0.1, 250 epochs)

In [9]:
def run_csi_epoch(model, loader, optimizer=None, criterion=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    preds_all, labels_all = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            total += y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            preds_all.append(out.argmax(1).detach().cpu().numpy())
            labels_all.append(y.detach().cpu().numpy())
    yt = np.concatenate(labels_all); yp = np.concatenate(preds_all)
    return total_loss/total, correct/total, f1_score(yt, yp, average="macro")

CSI_CKPT = CKPT_ROOT / "best_resnet18_csi.pth"
CSI_HISTORY = META_ROOT / "history_resnet18_csi.json"

criterion_csi = nn.CrossEntropyLoss(label_smoothing=CSI_LABEL_SMOOTHING)
optimizer_csi = torch.optim.AdamW(resnet18.parameters(), lr=CSI_LR, weight_decay=CSI_WEIGHT_DECAY)
scheduler_csi = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_csi, T_max=CSI_EPOCHS)

history_csi = {"train_loss":[], "train_acc":[], "train_f1":[], "val_loss":[], "val_acc":[], "val_f1":[], "lr":[]}
best_val_acc_csi = -1.0

if RUN_CSI:
    t_start = time.time()
    for epoch in range(1, CSI_EPOCHS + 1):
        t0 = time.time()
        tr = run_csi_epoch(resnet18, csi_train_loader, optimizer_csi, criterion_csi)
        va = run_csi_epoch(resnet18, csi_val_loader, None, criterion_csi)
        scheduler_csi.step()
        for k, v in zip(["train_loss","train_acc","train_f1"], tr): history_csi[k].append(v)
        for k, v in zip(["val_loss","val_acc","val_f1"], va): history_csi[k].append(v)
        history_csi["lr"].append(optimizer_csi.param_groups[0]["lr"])
        is_best = va[1] > best_val_acc_csi
        if is_best:
            best_val_acc_csi = va[1]
            torch.save({
                "model_state_dict": resnet18.state_dict(), "architecture": "ResNet18_CSI",
                "num_classes": len(CLASSES), "classes": CLASSES,
                "input_shape": [T_STEPS, F_SUB, CSI_TOTAL_CHANNELS],
                "preprocessing": {"amp_mean": AMP_MEAN, "amp_std": AMP_STD, "phase_divisor": float(np.pi),
                                   "phase_processing": "unwrap(axis=1) + linear trend removal",
                                   "antenna_processing": "mean(axis=-1)", "links": ["csi0","csi1","csi2"],
                                   "concat": "amplitude + phase"},
                "split_seed": SEED, "best_epoch": epoch,
                "best_val_acc": float(va[1]), "best_val_f1": float(va[2]),
            }, CSI_CKPT)
        dt = time.time() - t0
        if epoch == 1 or epoch % 10 == 0 or epoch == CSI_EPOCHS or is_best:
            print(f"[CSI] Epoch {epoch:03d}/{CSI_EPOCHS} | train acc={tr[1]:.4f} f1={tr[2]:.4f} | "
                  f"val acc={va[1]:.4f} f1={va[2]:.4f} | {dt:.1f}s" + (" \u2b50 BEST" if is_best else ""))
    with open(CSI_HISTORY, "w") as f:
        json.dump(history_csi, f, indent=2)
    print(f"\n\u2705 CSI termine en {(time.time()-t_start)/60:.1f} min | Best val acc: {best_val_acc_csi:.4f}")


[CSI] Epoch 001/250 | train acc=0.6730 f1=0.6721 | val acc=0.7512 f1=0.7529 | 60.8s ⭐ BEST
[CSI] Epoch 002/250 | train acc=0.7967 f1=0.7964 | val acc=0.8017 f1=0.8030 | 65.1s ⭐ BEST
[CSI] Epoch 003/250 | train acc=0.8389 f1=0.8382 | val acc=0.8458 f1=0.8486 | 65.8s ⭐ BEST
[CSI] Epoch 004/250 | train acc=0.8610 f1=0.8610 | val acc=0.8605 f1=0.8637 | 66.4s ⭐ BEST
[CSI] Epoch 005/250 | train acc=0.8822 f1=0.8820 | val acc=0.8773 f1=0.8809 | 65.8s ⭐ BEST
[CSI] Epoch 006/250 | train acc=0.8927 f1=0.8925 | val acc=0.8841 f1=0.8870 | 66.3s ⭐ BEST
[CSI] Epoch 008/250 | train acc=0.9117 f1=0.9116 | val acc=0.8886 f1=0.8909 | 66.7s ⭐ BEST
[CSI] Epoch 009/250 | train acc=0.9165 f1=0.9164 | val acc=0.8924 f1=0.8944 | 66.7s ⭐ BEST
[CSI] Epoch 010/250 | train acc=0.9234 f1=0.9236 | val acc=0.8871 f1=0.8898 | 66.1s
[CSI] Epoch 011/250 | train acc=0.9291 f1=0.9290 | val acc=0.9050 f1=0.9073 | 66.0s ⭐ BEST
[CSI] Epoch 013/250 | train acc=0.9359 f1=0.9358 | val acc=0.9053 f1=0.9078 | 66.1s ⭐ BEST
[CSI] 

## 9. Évaluation ResNet-18 CSI (best checkpoint)

In [10]:
ckpt_csi = torch.load(CSI_CKPT, map_location=DEVICE, weights_only=False)
resnet18.load_state_dict(ckpt_csi["model_state_dict"])
resnet18.eval()

yt_csi, yp_csi = [], []
with torch.no_grad():
    for x, y in csi_test_loader:
        out = resnet18(x.to(DEVICE, non_blocking=True))
        yp_csi.append(out.argmax(1).cpu().numpy())
        yt_csi.append(y.numpy())
yt_csi = np.concatenate(yt_csi); yp_csi = np.concatenate(yp_csi)

test_acc_csi = float((yt_csi == yp_csi).mean())
test_f1_csi = float(f1_score(yt_csi, yp_csi, average="macro"))
report_csi = classification_report(yt_csi, yp_csi, target_names=CLASSES, digits=4)
cm_csi = confusion_matrix(yt_csi, yp_csi)
print(report_csi)
print(f"Test accuracy: {test_acc_csi:.4f} | Test F1-macro: {test_f1_csi:.4f}")


              precision    recall  f1-score   support

        fall     0.9845    0.9725    0.9785      1635
      nfall1     0.9595    0.9680    0.9637      2250
      nfall2     0.9632    0.9632    0.9632      2145

    accuracy                         0.9675      6030
   macro avg     0.9691    0.9679    0.9684      6030
weighted avg     0.9676    0.9675    0.9675      6030

Test accuracy: 0.9675 | Test F1-macro: 0.9684


## 10. Export complet + ZIP ResNet-18 CSI, puis libération mémoire

Le zip est écrit directement à la racine de `/kaggle/working` pour apparaître immédiatement dans
l'onglet **Output** de la version commitée.

In [11]:
CSI_EXPORT_DIR = OUT_ROOT / "export_resnet18_csi"
CSI_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

csi_full_export = {
    "model_state_dict": resnet18.state_dict(),
    "architecture": "ResNet18_CSI",
    "n_params": int(n_params_resnet),
    "num_classes": len(CLASSES),
    "classes": CLASSES,
    "label_mapping": LABEL2IDX,
    "input_shape": [T_STEPS, F_SUB, CSI_TOTAL_CHANNELS],
    "preprocessing": ckpt_csi["preprocessing"],
    "hyperparameters": {"batch_size": CSI_BATCH_SIZE, "epochs": CSI_EPOCHS, "lr": CSI_LR,
                         "weight_decay": CSI_WEIGHT_DECAY, "label_smoothing": CSI_LABEL_SMOOTHING,
                         "dropout": CSI_DROPOUT, "optimizer": "AdamW", "scheduler": "CosineAnnealingLR"},
    "best_epoch": ckpt_csi["best_epoch"], "best_val_acc": ckpt_csi["best_val_acc"], "best_val_f1": ckpt_csi["best_val_f1"],
    "test_accuracy": test_acc_csi, "test_f1_macro": test_f1_csi,
    "confusion_matrix": cm_csi.tolist(), "classification_report": report_csi,
    "split_seed": SEED, "exported_at": datetime.now().isoformat(),
}
torch.save(csi_full_export, CSI_EXPORT_DIR / "resnet18_csi_full_export.pt")
shutil.copy(DEF_ROOT / "resnet18_csi.py", CSI_EXPORT_DIR / "resnet18_csi.py")
shutil.copy(META_ROOT / "csi_normalization.json", CSI_EXPORT_DIR / "csi_normalization.json")
shutil.copy(META_ROOT / "label_mapping.json", CSI_EXPORT_DIR / "label_mapping.json")
shutil.copy(META_ROOT / "paired_samples_with_split.csv", CSI_EXPORT_DIR / "paired_samples_final.csv")
with open(CSI_EXPORT_DIR / "test_classification_report.txt", "w") as f:
    f.write(report_csi)

CSI_ZIP = shutil.make_archive(str(FINAL_ROOT / "resnet18_csi_fusion_ready"), "zip", root_dir=CSI_EXPORT_DIR)
print(f"\u2705 Export CSI: {CSI_ZIP}")
print(f"   Test accuracy: {test_acc_csi:.4f} | Test F1-macro: {test_f1_csi:.4f}")

# --- Liberation memoire avant OpenPose ---
del optimizer_csi, scheduler_csi
del csi_train_loader, csi_val_loader, csi_test_loader
del train_csi, val_csi, test_csi, csi_sampler
del resnet18
gc.collect()
torch.cuda.empty_cache()
print(f"RAM apres nettoyage: {mem_gb():.2f} Go")
print("\u2705 Memoire CSI liberee -- pret pour MobileNetV4.")


✅ Export CSI: /kaggle/working/resnet18_csi_fusion_ready.zip
   Test accuracy: 0.9675 | Test F1-macro: 0.9684
RAM apres nettoyage: 4.92 Go
✅ Memoire CSI liberee -- pret pour MobileNetV4.


## 11. Prétraitement + cache OpenPose (mmap, `float16`)

Identique au notebook OpenPose fourni : `X = concatenate([aff, kpt], axis=1)` → `57 × 36 × 64`.

In [12]:
OP_CACHE = CACHE_ROOT / "openpose"
OP_CACHE.mkdir(parents=True, exist_ok=True)

pairs["op_cache_pos"] = -1
op_manifest = []
op_map2 = {file_number(p): p for p in OPENPOSE_FILES}

for n, group in pairs.groupby("file_number", sort=True):
    op_path = op_map2[int(n)]
    target_idx = group.index.tolist()
    target_rows = pairs.loc[target_idx, "openpose_row"].astype(int).tolist()

    x_path = OP_CACHE / f"X_{n}.npy"
    if not x_path.exists():
        print(f"Pretraitement OpenPose full_{n} ...")
        df = pd.read_pickle(op_path)
        aff_stack = np.stack([np.asarray(df.iloc[i]["aff"]) for i in target_rows]).astype(np.float16)
        kpt_stack = np.stack([np.asarray(df.iloc[i]["kpt"]) for i in target_rows]).astype(np.float16)
        X_file = np.concatenate([aff_stack, kpt_stack], axis=1)
        assert X_file.shape[1:] == (OP_IN_CHANNELS, OP_H, OP_W), X_file.shape
        np.save(x_path, X_file)
        del df, aff_stack, kpt_stack, X_file
        gc.collect()
    else:
        print(f"Cache OpenPose deja present: full_{n}")

    for pos, global_idx in enumerate(target_idx):
        pairs.loc[global_idx, "op_cache_pos"] = pos
    op_manifest.append({"file_number": int(n), "x_path": str(x_path), "n_samples": len(target_rows)})

assert (pairs["op_cache_pos"] >= 0).all()
with open(OP_CACHE / "manifest.pkl", "wb") as f:
    pickle.dump(op_manifest, f)
pairs.to_csv(META_ROOT / "paired_samples_final.csv", index=False)
print("OpenPose files caches:", len(op_manifest), "| RAM:", mem_gb(), "Go")


Pretraitement OpenPose full_0 ...
Pretraitement OpenPose full_1 ...
Pretraitement OpenPose full_2 ...
Pretraitement OpenPose full_3 ...
Pretraitement OpenPose full_4 ...
Pretraitement OpenPose full_5 ...
Pretraitement OpenPose full_6 ...
Pretraitement OpenPose full_7 ...
Pretraitement OpenPose full_8 ...
Pretraitement OpenPose full_9 ...
OpenPose files caches: 10 | RAM: 4.92378112 Go


## 12. Dataset + DataLoaders OpenPose (mmap caché en mémoire par fichier)

In [13]:
class OpenPoseFusionPrepDataset(Dataset):
    def __init__(self, pairs_df, split, op_manifest):
        self.df = pairs_df[pairs_df["split"] == split].reset_index(drop=True)
        self.manifest = {m["file_number"]: m for m in op_manifest}
        self.split = split
        self._mmap_cache = {}

    def _get_mmap(self, file_number):
        if file_number not in self._mmap_cache:
            self._mmap_cache[file_number] = np.load(self.manifest[file_number]["x_path"], mmap_mode="r")
        return self._mmap_cache[file_number]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = np.array(self._get_mmap(int(r.file_number))[int(r.op_cache_pos)], dtype=np.float32, copy=True)
        if self.split == "train":
            if random.random() < 0.5:
                x = x + np.random.normal(0, 0.03, x.shape).astype(np.float32)
            if random.random() < 0.3:
                n_drop = random.randint(1, 4)
                drop_idx = np.random.choice(x.shape[0], n_drop, replace=False)
                x[drop_idx] = 0.0
            if random.random() < 0.3:
                h_size, w_size = random.randint(4, 10), random.randint(6, 16)
                h0 = random.randint(0, OP_H - h_size); w0 = random.randint(0, OP_W - w_size)
                x[:, h0:h0+h_size, w0:w0+w_size] = 0.0
        return torch.from_numpy(x), torch.tensor(int(r.label_id), dtype=torch.long)

train_op = OpenPoseFusionPrepDataset(pairs, "train", op_manifest)
val_op = OpenPoseFusionPrepDataset(pairs, "val", op_manifest)
test_op = OpenPoseFusionPrepDataset(pairs, "test", op_manifest)

op_sampler = make_sampler(pairs)
OP_WORKERS = min(4, os.cpu_count() or 2)
op_train_loader = DataLoader(train_op, batch_size=OP_BATCH_SIZE, sampler=op_sampler,
                              num_workers=OP_WORKERS, pin_memory=True, drop_last=True,
                              persistent_workers=(OP_WORKERS > 0))
op_val_loader = DataLoader(val_op, batch_size=OP_BATCH_SIZE, shuffle=False,
                            num_workers=OP_WORKERS, pin_memory=True, persistent_workers=(OP_WORKERS > 0))
op_test_loader = DataLoader(test_op, batch_size=OP_BATCH_SIZE, shuffle=False,
                             num_workers=OP_WORKERS, pin_memory=True, persistent_workers=(OP_WORKERS > 0))
print("OP:", len(train_op), len(val_op), len(test_op))


OP: 28140 6030 6030


## 13. MobileNetV4 OpenPose — architecture identique au notebook fourni (timm + CBAM)

In [14]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "timm"], check=True)
    import timm

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False), nn.ReLU(inplace=True),
                                  nn.Conv2d(hidden, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        x = x * self.ca(x)
        return x * self.sa(x)

class GenericFallNet(nn.Module):
    def __init__(self, backbone_name="mobilenetv4_conv_small", in_channels=57, num_classes=3, dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, in_chans=in_channels,
                                           num_classes=0, global_pool="")
        with torch.no_grad():
            feat = self.backbone(torch.zeros(1, in_channels, 36, 64))
        feat_channels = feat.shape[1]
        self.cbam = CBAM(feat_channels)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_channels, 128),
                                         nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.backbone(x)
        x = self.cbam(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

mobilenetv4 = GenericFallNet("mobilenetv4_conv_small", in_channels=OP_IN_CHANNELS,
                              num_classes=len(CLASSES), dropout=OP_DROPOUT).to(DEVICE)
n_params_mnv4 = sum(p.numel() for p in mobilenetv4.parameters())
print(f"MobileNetV4 OpenPose parametres: {n_params_mnv4:,}")

MNV4_SRC = '''import torch
import torch.nn as nn
import timm

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False), nn.ReLU(inplace=True),
                                  nn.Conv2d(hidden, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        x = x * self.ca(x)
        return x * self.sa(x)

class GenericFallNet(nn.Module):
    def __init__(self, backbone_name="mobilenetv4_conv_small", in_channels=57, num_classes=3, dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, in_chans=in_channels,
                                           num_classes=0, global_pool="")
        with torch.no_grad():
            feat = self.backbone(torch.zeros(1, in_channels, 36, 64))
        feat_channels = feat.shape[1]
        self.cbam = CBAM(feat_channels)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_channels, 128),
                                         nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.backbone(x)
        x = self.cbam(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)
'''
with open(DEF_ROOT / "mobilenetv4_openpose.py", "w") as f:
    f.write(MNV4_SRC)
print("Definition exportee:", DEF_ROOT / "mobilenetv4_openpose.py")


MobileNetV4 OpenPose parametres: 2,877,829
Definition exportee: /kaggle/working/fusion_preparation/definitions/mobilenetv4_openpose.py


## 14. Entraînement MobileNetV4 OpenPose (Adam, MixUp, AMP, cosine annealing, 150 epochs)

In [15]:
def mixup_batch(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam

def run_op_epoch(model, loader, optimizer, criterion, scaler, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, n_samples = 0.0, 0
    preds_all, labels_all = [], []
    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            if train_mode:
                x_mix, ya, yb, lam = mixup_batch(x, y, OP_MIXUP_ALPHA)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    out = model(x_mix)
                    loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(optimizer); scaler.update()
            else:
                with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    out = model(x); loss = criterion(out, y)
            bs = y.size(0)
            total_loss += loss.item() * bs; n_samples += bs
            preds_all.append(out.argmax(1).detach().cpu().numpy())
            labels_all.append(y.detach().cpu().numpy())
    yt = np.concatenate(labels_all); yp = np.concatenate(preds_all)
    return total_loss/n_samples, (yt == yp).mean(), f1_score(yt, yp, average="macro")

OP_CKPT = CKPT_ROOT / "best_mobilenetv4_openpose.pth"
OP_HISTORY = META_ROOT / "history_mobilenetv4_openpose.json"

criterion_op = nn.CrossEntropyLoss(label_smoothing=OP_LABEL_SMOOTHING)
optimizer_op = torch.optim.Adam(mobilenetv4.parameters(), lr=OP_LR, weight_decay=OP_WEIGHT_DECAY)
scheduler_op = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_op, T_max=OP_EPOCHS)
scaler_op = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

history_op = {"train_loss":[], "train_acc":[], "train_f1":[], "val_loss":[], "val_acc":[], "val_f1":[], "lr":[]}
best_val_f1_op = -1.0

if RUN_OPENPOSE:
    t_start = time.time()
    for epoch in range(1, OP_EPOCHS + 1):
        t0 = time.time()
        tr = run_op_epoch(mobilenetv4, op_train_loader, optimizer_op, criterion_op, scaler_op, True)
        va = run_op_epoch(mobilenetv4, op_val_loader, optimizer_op, criterion_op, scaler_op, False)
        scheduler_op.step()
        for k, v in zip(["train_loss","train_acc","train_f1"], tr): history_op[k].append(v)
        for k, v in zip(["val_loss","val_acc","val_f1"], va): history_op[k].append(v)
        history_op["lr"].append(optimizer_op.param_groups[0]["lr"])
        is_best = va[2] > best_val_f1_op
        if is_best:
            best_val_f1_op = va[2]
            torch.save({
                "model_state_dict": mobilenetv4.state_dict(), "architecture": "MobileNetV4_OpenPose",
                "backbone_name": "mobilenetv4_conv_small", "num_classes": len(CLASSES), "classes": CLASSES,
                "input_shape": [OP_IN_CHANNELS, OP_H, OP_W],
                "preprocessing": {"representation": "concatenate aff + kpt", "aff_channels": N_AFF,
                                   "kpt_channels": N_KPT, "shape": [OP_IN_CHANNELS, OP_H, OP_W], "normalization": "none"},
                "split_seed": SEED, "best_epoch": epoch,
                "best_val_acc": float(va[1]), "best_val_f1": float(va[2]),
            }, OP_CKPT)
        dt = time.time() - t0
        if epoch == 1 or epoch % 10 == 0 or epoch == OP_EPOCHS or is_best:
            print(f"[OpenPose] Epoch {epoch:03d}/{OP_EPOCHS} | train acc={tr[1]:.4f} f1={tr[2]:.4f} | "
                  f"val acc={va[1]:.4f} f1={va[2]:.4f} | {dt:.1f}s" + (" \u2b50 BEST" if is_best else ""))
    with open(OP_HISTORY, "w") as f:
        json.dump(history_op, f, indent=2)
    print(f"\n\u2705 OpenPose termine en {(time.time()-t_start)/60:.1f} min | Best val F1: {best_val_f1_op:.4f}")


[OpenPose] Epoch 001/250 | train acc=0.5035 f1=0.4991 | val acc=0.7448 f1=0.7437 | 51.0s ⭐ BEST
[OpenPose] Epoch 002/250 | train acc=0.5426 f1=0.5423 | val acc=0.8385 f1=0.8392 | 41.7s ⭐ BEST
[OpenPose] Epoch 003/250 | train acc=0.6079 f1=0.6079 | val acc=0.8813 f1=0.8809 | 41.0s ⭐ BEST
[OpenPose] Epoch 004/250 | train acc=0.6088 f1=0.6088 | val acc=0.8839 f1=0.8836 | 41.3s ⭐ BEST
[OpenPose] Epoch 005/250 | train acc=0.6191 f1=0.6190 | val acc=0.8998 f1=0.8994 | 41.2s ⭐ BEST
[OpenPose] Epoch 007/250 | train acc=0.6459 f1=0.6458 | val acc=0.9176 f1=0.9181 | 41.2s ⭐ BEST
[OpenPose] Epoch 008/250 | train acc=0.6627 f1=0.6626 | val acc=0.9247 f1=0.9254 | 41.2s ⭐ BEST
[OpenPose] Epoch 010/250 | train acc=0.6241 f1=0.6241 | val acc=0.9335 f1=0.9338 | 41.2s ⭐ BEST
[OpenPose] Epoch 012/250 | train acc=0.6460 f1=0.6459 | val acc=0.9357 f1=0.9354 | 41.5s ⭐ BEST
[OpenPose] Epoch 014/250 | train acc=0.6136 f1=0.6136 | val acc=0.9436 f1=0.9445 | 41.2s ⭐ BEST
[OpenPose] Epoch 015/250 | train acc=0.6

## 15. Évaluation MobileNetV4 OpenPose (best checkpoint)

In [16]:
ckpt_op = torch.load(OP_CKPT, map_location=DEVICE, weights_only=False)
mobilenetv4.load_state_dict(ckpt_op["model_state_dict"])
mobilenetv4.eval()

yt_op, yp_op = [], []
with torch.no_grad():
    for x, y in op_test_loader:
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            out = mobilenetv4(x.to(DEVICE, non_blocking=True))
        yp_op.append(out.argmax(1).cpu().numpy())
        yt_op.append(y.numpy())
yt_op = np.concatenate(yt_op); yp_op = np.concatenate(yp_op)

test_acc_op = float((yt_op == yp_op).mean())
test_f1_op = float(f1_score(yt_op, yp_op, average="macro"))
report_op = classification_report(yt_op, yp_op, target_names=CLASSES, digits=4)
cm_op = confusion_matrix(yt_op, yp_op)
print(report_op)
print(f"Test accuracy: {test_acc_op:.4f} | Test F1-macro: {test_f1_op:.4f}")


              precision    recall  f1-score   support

        fall     0.9877    0.9804    0.9840      1635
      nfall1     0.9902    0.9844    0.9873      2250
      nfall2     0.9756    0.9869    0.9812      2145

    accuracy                         0.9842      6030
   macro avg     0.9845    0.9839    0.9842      6030
weighted avg     0.9843    0.9842    0.9843      6030

Test accuracy: 0.9842 | Test F1-macro: 0.9842


## 16. Export complet + ZIP MobileNetV4 OpenPose, puis libération mémoire

In [17]:
OP_EXPORT_DIR = OUT_ROOT / "export_mobilenetv4_openpose"
OP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

op_full_export = {
    "model_state_dict": mobilenetv4.state_dict(),
    "architecture": "MobileNetV4_OpenPose", "backbone_name": "mobilenetv4_conv_small",
    "n_params": int(n_params_mnv4), "num_classes": len(CLASSES), "classes": CLASSES,
    "label_mapping": LABEL2IDX, "input_shape": [OP_IN_CHANNELS, OP_H, OP_W],
    "preprocessing": ckpt_op["preprocessing"],
    "hyperparameters": {"batch_size": OP_BATCH_SIZE, "epochs": OP_EPOCHS, "lr": OP_LR,
                         "weight_decay": OP_WEIGHT_DECAY, "label_smoothing": OP_LABEL_SMOOTHING,
                         "dropout": OP_DROPOUT, "mixup_alpha": OP_MIXUP_ALPHA,
                         "optimizer": "Adam", "scheduler": "CosineAnnealingLR"},
    "best_epoch": ckpt_op["best_epoch"], "best_val_acc": ckpt_op["best_val_acc"], "best_val_f1": ckpt_op["best_val_f1"],
    "test_accuracy": test_acc_op, "test_f1_macro": test_f1_op,
    "confusion_matrix": cm_op.tolist(), "classification_report": report_op,
    "split_seed": SEED, "exported_at": datetime.now().isoformat(),
}
torch.save(op_full_export, OP_EXPORT_DIR / "mobilenetv4_openpose_full_export.pt")
shutil.copy(DEF_ROOT / "mobilenetv4_openpose.py", OP_EXPORT_DIR / "mobilenetv4_openpose.py")
shutil.copy(META_ROOT / "label_mapping.json", OP_EXPORT_DIR / "label_mapping.json")
shutil.copy(META_ROOT / "paired_samples_final.csv", OP_EXPORT_DIR / "paired_samples_final.csv")
with open(OP_EXPORT_DIR / "test_classification_report.txt", "w") as f:
    f.write(report_op)

OP_ZIP = shutil.make_archive(str(FINAL_ROOT / "mobilenetv4_openpose_fusion_ready"), "zip", root_dir=OP_EXPORT_DIR)
print(f"\u2705 Export OpenPose: {OP_ZIP}")
print(f"   Test accuracy: {test_acc_op:.4f} | Test F1-macro: {test_f1_op:.4f}")

del optimizer_op, scheduler_op, scaler_op
del op_train_loader, op_val_loader, op_test_loader
del train_op, val_op, test_op, op_sampler
del mobilenetv4
gc.collect()
torch.cuda.empty_cache()
print(f"RAM finale: {mem_gb():.2f} Go")


✅ Export OpenPose: /kaggle/working/mobilenetv4_openpose_fusion_ready.zip
   Test accuracy: 0.9842 | Test F1-macro: 0.9842
RAM finale: 6.19 Go


## 17. Validation finale — ces 2 fichiers sont ce qu'il vous faut demain

Après le commit ("Save & Run All"), retrouvez-les dans l'onglet **Output** de la version terminée :

In [18]:
print("Fichiers finaux dans /kaggle/working :\n")
for fname in ["resnet18_csi_fusion_ready.zip", "mobilenetv4_openpose_fusion_ready.zip"]:
    p = FINAL_ROOT / fname
    size_mb = p.stat().st_size / 1e6 if p.exists() else 0
    print(f"  {'\u2705' if p.exists() else '\u274c MANQUANT'} {fname} ({size_mb:.1f} MB)")

print("\nResume:")
print(f"  ResNet-18 CSI       -> test_acc={test_acc_csi:.4f}  test_f1={test_f1_csi:.4f}")
print(f"  MobileNetV4 OpenPose -> test_acc={test_acc_op:.4f}  test_f1={test_f1_op:.4f}")
print("\n\u2705 Pret pour la fusion. Telechargez les 2 zips depuis l'onglet Output de cette version.")


Fichiers finaux dans /kaggle/working :

  ✅ resnet18_csi_fusion_ready.zip (41.9 MB)
  ✅ mobilenetv4_openpose_fusion_ready.zip (5.5 MB)

Resume:
  ResNet-18 CSI       -> test_acc=0.9675  test_f1=0.9684
  MobileNetV4 OpenPose -> test_acc=0.9842  test_f1=0.9842

✅ Pret pour la fusion. Telechargez les 2 zips depuis l'onglet Output de cette version.
